In [1]:
"""
Bengali RAG Hallucination Detection
=====================================
Task: Given (prompt_bn, response_bn, context[optional]) -> predict
      label = 1 (faithful) or 0 (hallucinated)
Metric: Binary F1 on hallucinated class (label=0)

APPROACH (hybrid, NOT plain LSTM/XGBoost-on-text):
  1. Encode context/prompt/response with a multilingual sentence embedding
     model (works well for Bangla) -> similarity features.
  2. Use a pretrained multilingual NLI (entailment) model to score whether
     the response is *entailed* by the context (or by the prompt, when
     context is missing) -> the single strongest hallucination signal.
  3. Build a small tabular feature set from (1)+(2)+lexical overlap stats.
  4. Train a gradient-boosted tree (XGBoost/LightGBM) on those features.
     This is far more data-efficient than training an RNN/Transformer
     from scratch on a few hundred/thousand rows, and much stronger than
     bag-of-words + XGBoost because the NLI/embedding features already
     encode semantic entailment.

Install (Kaggle notebook):
    pip install -U sentence-transformers transformers xgboost scikit-learn
"""

import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"   # given train file (json)
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"            # given test file (csv, no label column)
SAMPLE_SUB_PATH = "/kaggle/input/competitions/bengali-hallucination/sample submission.csv"  # has columns: id, label
SUBMISSION_OUT = "submission.csv"

EMBED_MODEL_NAME = "sentence-transformers/LaBSE"          # strong for Bangla
# If Kaggle internet is OFF, download once elsewhere and attach as a Dataset,
# then point this to the local path instead, e.g.:
# EMBED_MODEL_NAME = "/kaggle/input/bengali-hallucination-models/labse_model"

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"  # multilingual NLI
# Same offline fix:
# NLI_MODEL_NAME = "/kaggle/input/bengali-hallucination-models/nli_model"

RANDOM_STATE = 42
N_FOLDS = 5

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
def load_json(path):
    """Load the train file (JSON list of dicts with a label column)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df


def load_test_csv(path):
    """
    Load the Kaggle test file (CSV, no label column).
    Expected columns: id, context, prompt_bn, response_bn
    (context may be missing / empty / literal "[NULL]" string)
    """
    df = pd.read_csv(path)

    # normalize column names just in case of casing/whitespace differences
    df.columns = [c.strip() for c in df.columns]

    # make sure required columns exist
    required = {"id", "prompt_bn", "response_bn"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"test_set.csv is missing expected columns: {missing}. "
                          f"Found columns: {list(df.columns)}")

    if "context" not in df.columns:
        df["context"] = np.nan

    df["context"] = df["context"].replace("[NULL]", np.nan)
    df["context"] = df["context"].replace("", np.nan)
    df["has_context"] = df["context"].notna().astype(int)
    return df


# ----------------------------------------------------------------------
# 2. EMBEDDING MODEL (semantic similarity features)
# ----------------------------------------------------------------------
from sentence_transformers import SentenceTransformer, util

class Embedder:
    def __init__(self, model_name=EMBED_MODEL_NAME, device=None):
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, texts):
        clean = []
        for t in texts:
            if t is None or (isinstance(t, float) and np.isnan(t)):
                clean.append("।")
            else:
                t = str(t)
                clean.append(t if t.strip() else "।")
        return self.model.encode(
            clean, batch_size=64, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True
        )

    def cosine(self, a_emb, b_emb):
        return np.sum(a_emb * b_emb, axis=1)  # already normalized


# ----------------------------------------------------------------------
# 3. NLI ENTAILMENT MODEL (the core hallucination signal)
# ----------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

class NLIScorer:
    """
    premise  = context (or prompt, if context is missing)
    hypothesis = response
    Returns P(entailment), P(neutral), P(contradiction)
    """
    def __init__(self, model_name=NLI_MODEL_NAME, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        # mDeBERTa-mnli-xnli label order:
        self.label_names = ["entailment", "neutral", "contradiction"]

    @torch.no_grad()
    def score_batch(self, premises, hypotheses, batch_size=32):
        # force everything to plain, non-empty strings (NaN/float/None -> "")
        premises = ["" if p is None or (isinstance(p, float) and np.isnan(p)) else str(p) for p in premises]
        hypotheses = ["" if h is None or (isinstance(h, float) and np.isnan(h)) else str(h) for h in hypotheses]
        premises = [p if p.strip() else "।" for p in premises]      # tokenizer needs non-empty text
        hypotheses = [h if h.strip() else "।" for h in hypotheses]

        all_probs = []
        for i in range(0, len(premises), batch_size):
            p_batch = premises[i:i + batch_size]
            h_batch = hypotheses[i:i + batch_size]
            inputs = self.tokenizer(
                p_batch, h_batch, truncation=True, padding=True,
                max_length=256, return_tensors="pt"
            ).to(self.device)
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)


# ----------------------------------------------------------------------
# 4. LEXICAL / SURFACE FEATURES (cheap but useful signal)
# ----------------------------------------------------------------------
import difflib

def bn_tokenize(text):
    if not isinstance(text, str):
        if text is None or (isinstance(text, float) and np.isnan(text)):
            return []
        text = str(text)
    return re.findall(r"[\u0980-\u09FF]+|\S+", text)

def token_overlap_ratio(a, b):
    ta, tb = set(bn_tokenize(a)), set(bn_tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def response_in_context(response, context):
    if not isinstance(context, str) or not isinstance(response, str):
        return 0
    return int(response.strip() in context)

def length_ratio(a, b):
    la, lb = len(bn_tokenize(a)), len(bn_tokenize(b))
    if lb == 0:
        return 0.0
    return la / lb

def fuzzy_ratio(a, b):
    a = "" if not isinstance(a, str) else a
    b = "" if not isinstance(b, str) else b
    if not a or not b:
        return 0.0
    return difflib.SequenceMatcher(None, a, b).ratio()

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def extract_digits(text):
    if not isinstance(text, str):
        return set()
    text = text.translate(BN_DIGITS)
    return set(re.findall(r"\d+", text))

def digit_match_feature(response, context_or_prompt):
    r_digits = extract_digits(response)
    c_digits = extract_digits(context_or_prompt)
    if not r_digits:
        return -1.0  # no digits in response -> not applicable
    return float(len(r_digits & c_digits) > 0)

def make_declarative_hypothesis(prompt, response):
    """
    Turn (question, short-answer) into a claim-like sentence so the NLI
    model actually knows WHAT is being asserted, e.g.:
      prompt   = 'অভ্র কিবোর্ড কে উদ্ভাবন করেন ?'
      response = 'মেহদী হাসান খান'
      ->  'প্রশ্নঃ অভ্র কিবোর্ড কে উদ্ভাবন করেন ? উত্তরঃ মেহদী হাসান খান'
    This gives the entailment model a self-contained claim to verify
    against the premise (context or prompt), instead of a bare answer
    span with no question attached.
    """
    prompt = "" if not isinstance(prompt, str) else prompt.strip()
    response = "" if not isinstance(response, str) else response.strip()
    return f"প্রশ্নঃ {prompt} উত্তরঃ {response}"


# ----------------------------------------------------------------------
# 5. FEATURE ENGINEERING PIPELINE
# ----------------------------------------------------------------------
USE_DECLARATIVE_HYPOTHESIS = False  # <-- toggle: True = "প্রশ্নঃ...উত্তরঃ..." framing,
                                     #     False = raw response only (simpler, safer default)

def build_features(df, embedder, nli_scorer):
    df = df.copy()

    prompt_col = df["prompt_bn"]
    response_col = df["response_bn"].apply(lambda x: "" if not isinstance(x, str) else x)
    context_col = df["context"]

    # Hypothesis text fed to the NLI model as the "claim" being checked.
    if USE_DECLARATIVE_HYPOTHESIS:
        hypothesis = [
            make_declarative_hypothesis(p, r) for p, r in zip(prompt_col, response_col)
        ]
    else:
        hypothesis = response_col.tolist()

    # --- NLI vs CONTEXT (falls back to prompt as premise when context is missing) ---
    premise_ctx = context_col.fillna(prompt_col)
    probs_ctx = nli_scorer.score_batch(premise_ctx.tolist(), hypothesis)
    df["nli_ctx_entail"] = probs_ctx[:, 0]
    df["nli_ctx_neutral"] = probs_ctx[:, 1]
    df["nli_ctx_contra"] = probs_ctx[:, 2]

    # --- NLI vs PROMPT ALONE (self-consistency: catches on-topic-sounding but
    #     off-topic answers, computed for every row including has-context ones) ---
    probs_prompt = nli_scorer.score_batch(prompt_col.tolist(), hypothesis)
    df["nli_prompt_entail"] = probs_prompt[:, 0]
    df["nli_prompt_contra"] = probs_prompt[:, 2]

    # --- Embedding similarity features ---
    ctx_or_prompt_emb = embedder.encode(premise_ctx.tolist())
    resp_emb = embedder.encode(response_col.tolist())
    prompt_emb = embedder.encode(prompt_col.tolist())

    df["sim_premise_response"] = embedder.cosine(ctx_or_prompt_emb, resp_emb)
    df["sim_prompt_response"] = embedder.cosine(prompt_emb, resp_emb)

    # --- Lexical / surface features ---
    df["token_overlap_ctx_resp"] = [
        token_overlap_ratio(p, r) for p, r in zip(premise_ctx, response_col)
    ]
    df["response_substring_of_context"] = [
        response_in_context(r, c) for r, c in zip(response_col, context_col)
    ]
    df["fuzzy_ratio_ctx_resp"] = [
        fuzzy_ratio(p, r) for p, r in zip(premise_ctx, response_col)
    ]
    df["digit_match"] = [
        digit_match_feature(r, p) for r, p in zip(response_col, premise_ctx)
    ]
    df["len_ratio_resp_prompt"] = [
        length_ratio(r, p) for r, p in zip(response_col, prompt_col)
    ]
    df["response_len_tokens"] = response_col.apply(lambda x: len(bn_tokenize(x)))
    df["premise_is_context"] = df["has_context"]

    feature_cols = [
        "nli_ctx_entail", "nli_ctx_neutral", "nli_ctx_contra",
        "nli_prompt_entail", "nli_prompt_contra",
        "sim_premise_response", "sim_prompt_response",
        "token_overlap_ctx_resp", "response_substring_of_context",
        "fuzzy_ratio_ctx_resp", "digit_match",
        "len_ratio_resp_prompt", "response_len_tokens",
        "has_context", "premise_is_context",
    ]
    return df, feature_cols


# ----------------------------------------------------------------------
# 6. TRAIN + CROSS VALIDATE (XGBoost)
# ----------------------------------------------------------------------
def find_best_threshold(y_true, y_prob):
    """Search thresholds to maximize F1 on the hallucinated class (label=0)."""
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.05, 0.96, 0.01):
        preds = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, preds, pos_label=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def train_cv(df, feature_cols, target_col="label", n_folds=N_FOLDS):
    X = df[feature_cols].values
    y = df[target_col].values

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_probs = np.zeros(len(df))
    models = []
    fold_thresholds = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # NOTE: no scale_pos_weight -- classes are only mildly imbalanced
        # (roughly 45/55 split); forcing reweighting here hurt F1 in practice.
        model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_lambda=2.0,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        val_prob = model.predict_proba(X_val)[:, 1]
        oof_probs[val_idx] = val_prob

        # tune threshold WITHIN this fold's validation split only,
        # just to report per-fold quality (not used for the final threshold)
        t_fold, f1_fold = find_best_threshold(y_val, val_prob)
        fold_thresholds.append(t_fold)
        print(f"Fold {fold}: F1@0.5={f1_score(y_val,(val_prob>=0.5).astype(int),pos_label=0):.4f}"
              f"  |  F1@best({t_fold:.2f})={f1_fold:.4f}")
        models.append(model)

    # Final threshold = average of per-fold best thresholds (more stable than
    # picking a single global optimum on the full OOF vector, which can
    # overfit a small dataset).
    best_t = float(np.mean(fold_thresholds))
    oof_preds_tuned = (oof_probs >= best_t).astype(int)
    overall_f1 = f1_score(y, oof_preds_tuned, pos_label=0)
    print(f"\nAveraged threshold: {best_t:.2f}  ->  OOF hallucination-F1 = {overall_f1:.4f}")
    print(classification_report(y, oof_preds_tuned, target_names=["hallucinated(0)", "faithful(1)"]))

    # feature importance -- use this to prune noisy features if F1 is still low
    importances = np.mean([m.feature_importances_ for m in models], axis=0)
    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nFeature importances (averaged across folds):")
    print(imp_df.to_string(index=False))

    return models, best_t


# ----------------------------------------------------------------------
# 7. INFERENCE ON TEST SET
# ----------------------------------------------------------------------
def predict_test(models, df_test, feature_cols, threshold=0.5):
    X_test = df_test[feature_cols].values
    # Average predicted probabilities across folds, then apply the TUNED threshold
    probs = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)
    preds = (probs >= threshold).astype(int)
    return preds, probs


# ----------------------------------------------------------------------
# 8. MAIN
# ----------------------------------------------------------------------
if __name__ == "__main__":
    print("Loading data...")
    train_df = load_json(TRAIN_PATH)

    print("Loading embedding model...")
    embedder = Embedder()

    print("Loading NLI model...")
    nli_scorer = NLIScorer()

    print("Building features for train set...")
    train_df, feature_cols = build_features(train_df, embedder, nli_scorer)

    print("Training with 5-fold CV...")
    models, best_t = train_cv(train_df, feature_cols)

    # ---- Inference on the real test set ----
    import os
    if os.path.exists(TEST_PATH):
        print("Loading & featurizing test set...")
        test_df = load_test_csv(TEST_PATH)          # <-- CSV loader
        test_df, _ = build_features(test_df, embedder, nli_scorer)
        preds, probs = predict_test(models, test_df, feature_cols, threshold=best_t)

        # Build submission matching sample_submission.csv format: id, label
        submission = pd.DataFrame({
            "id": test_df["id"].values,
            "label": preds,
        })

        # sanity check against sample_submission.csv if available
        if os.path.exists(SAMPLE_SUB_PATH):
            sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
            assert list(submission.columns) == list(sample_sub.columns), \
                f"Column mismatch! Expected {list(sample_sub.columns)}, got {list(submission.columns)}"
            assert len(submission) == len(sample_sub), \
                f"Row count mismatch! Expected {len(sample_sub)}, got {len(submission)}"
            # keep the same id ordering as sample_submission
            submission = sample_sub[["id"]].merge(submission, on="id", how="left")

        submission.to_csv(SUBMISSION_OUT, index=False)
        print(f"Saved {SUBMISSION_OUT} with {len(submission)} rows")
        print(submission.head())

Loading data...
Loading embedding model...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]